# Trabajo Práctico N°2 — Predicción Final

**Alumno:** Gonzalo Zarazaga

---

Este notebook genera las predicciones finales sobre el dataset de entrega (sin etiquetas).

**Modelo seleccionado:** `xgboost_optimized.joblib`  
**Umbral de decisión:** calculado en `05_validacion.ipynb` (threshold que maximiza F1 en test)  
**Insumo:** `data/processed/smoking_predict_processed.csv`  
**Salida:** `data/processed/smoking_predictions.csv` — columnas `ID` y `smoking_prediction` (valores 0 ó 1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

PATH_PROC   = Path("../data/processed")
PATH_MODELS = Path("../models")

RANDOM_STATE = 42
TEST_SIZE    = 0.2

## 1. Selección del modelo final

Durante el proceso de experimentación se generaron dos modelos XGBoost:

| Modelo | Parámetros | F1-Test | Proceso |
|---|---|---|---|
| `xgboost_baseline.joblib` | n=100, depth=6, lr=0.1 | 0.6947 | Configuración inicial de referencia |
| `xgboost_optimized.joblib` | n=100, depth=6, lr=0.1 | 0.6947 | **GridSearchCV (cv=3)** confirmó estos parámetros como óptimos |

Los parámetros son idénticos porque la búsqueda por GridSearchCV confirmó que el baseline ya se encontraba
en el punto óptimo del espacio explorado (`n_estimators` × `max_depth`).

**Se elige `xgboost_optimized.joblib`** por las siguientes razones:
1. Es el modelo que pasó por el proceso de validación cruzada (cv=3), por lo que sus parámetros están respaldados experimentalmente
2. Representa la decisión final del pipeline de entrenamiento, no un punto de partida
3. En caso de que exploración futura cambie los parámetros, el modelo optimizado es el que reflejaría ese resultado

**Umbral de decisión:** en lugar del umbral default de 0.50, se usa el umbral calculado en `05_validacion.ipynb`
que maximiza el F1-Score sobre el conjunto de test. El análisis de threshold tuning mostró que bajar
el umbral incrementa significativamente el recall (detección de fumadores) con una mejora neta en F1.

In [ ]:
model = joblib.load(PATH_MODELS / "xgboost_optimized.joblib")

with open(PATH_MODELS / "best_params.json") as f:
    best_params = json.load(f)

print(f"Modelo: {type(model).__name__}")
print(f"Parámetros: {best_params}")

## 2. Determinación del umbral óptimo

El umbral se calcula sobre el conjunto de test (80/20 split del dataset etiquetado),
buscando el valor que maximiza el F1-Score para la clase 1 (fumador).

Este cálculo replica lo realizado en `05_validacion.ipynb` para tener el valor exacto disponible en este notebook.

In [ ]:
df_train_full = pd.read_csv(PATH_PROC / "smoking_train_processed.csv")
X_full = df_train_full.drop(columns=["smoking"])
y_full = df_train_full["smoking"]

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_full
)

proba_test = model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.05, 0.96, 0.01)
res = [
    {
        "threshold": t,
        "f1":        f1_score(y_test, (proba_test >= t).astype(int), zero_division=0),
        "precision": precision_score(y_test, (proba_test >= t).astype(int), zero_division=0),
        "recall":    recall_score(y_test, (proba_test >= t).astype(int), zero_division=0),
    }
    for t in thresholds
]
df_thresh = pd.DataFrame(res)
idx_mejor    = df_thresh["f1"].idxmax()
THRESHOLD    = df_thresh.loc[idx_mejor, "threshold"]
f1_optimo    = df_thresh.loc[idx_mejor, "f1"]
f1_default   = f1_score(y_test, model.predict(X_test))

print(f"Umbral default (0.50):     F1 = {f1_default:.4f}")
print(f"Umbral óptimo  ({THRESHOLD:.2f}):     F1 = {f1_optimo:.4f}")
print(f"Mejora:                        +{(f1_optimo - f1_default)*100:.2f} pp")
print()
print(f"→ Se usará umbral = {THRESHOLD:.2f} para la predicción final")

## 3. Generación de predicciones

Se carga el dataset de entrega (ya preprocesado en `03_preprocesamiento.ipynb`) y se generan
las predicciones aplicando el umbral óptimo sobre las probabilidades del modelo.

In [ ]:
df_predict = pd.read_csv(PATH_PROC / "smoking_predict_processed.csv")

print(f"Dataset de entrega: {df_predict.shape}")
print(f"Columnas: {list(df_predict.columns)}")
df_predict.head(3)

In [ ]:
IDs      = df_predict["ID"]
X_pred   = df_predict.drop(columns=["ID"])

# Verificar que las features coincidan con las del modelo
features_modelo = X_full.columns.tolist()
features_pred   = X_pred.columns.tolist()
assert features_modelo == features_pred, \
    f"Mismatch de features: {set(features_modelo) ^ set(features_pred)}"
print(f"✓ Features alineadas: {len(features_modelo)} columnas")

# Probabilidades y predicciones con umbral óptimo
proba_pred = model.predict_proba(X_pred)[:, 1]
y_pred     = (proba_pred >= THRESHOLD).astype(int)

print(f"✓ Predicciones generadas: {len(y_pred)} registros")
print(f"  Umbral aplicado: {THRESHOLD:.2f}")
print(f"  Predichos como fumadores (1): {y_pred.sum()} ({y_pred.mean()*100:.1f}%)")
print(f"  Predichos como no fumadores (0): {(y_pred==0).sum()} ({(y_pred==0).mean()*100:.1f}%)")

## 4. Verificación de la distribución de predicciones

Se compara la proporción de fumadores predicha contra la distribución real del dataset de entrenamiento.
Si el dataset de entrega es representativo (como se verificó en `04_entrenamiento_y_optimizacion.ipynb`),
las proporciones deberían ser similares.

In [ ]:
prop_train   = y_full.mean()
prop_pred    = y_pred.mean()
prop_test_50 = model.predict(X_pred).mean()

print("Proporción de fumadores predicha:")
print(f"  Train (real):                   {prop_train*100:.1f}%")
print(f"  Entrega (umbral 0.50, default): {prop_test_50*100:.1f}%")
print(f"  Entrega (umbral {THRESHOLD:.2f}, óptimo): {prop_pred*100:.1f}%")
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel izquierdo: barras comparativas
etiquetas = ["Train\n(real)", f"Predicción\n(t=0.50)", f"Predicción\n(t={THRESHOLD:.2f})"]
valores   = [prop_train, prop_test_50, prop_pred]
colores   = ["#3498db", "#95a5a6", "#2ecc71"]

bars = axes[0].bar(etiquetas, valores, color=colores, edgecolor="white", width=0.5)
for bar, val in zip(bars, valores):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.008,
                 f"{val*100:.1f}%",
                 ha="center", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Proporción de fumadores", fontsize=11)
axes[0].set_ylim(0, 0.7)
axes[0].set_title("Distribución de fumadores: train vs predicción",
                  fontsize=11, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)

# Panel derecho: histograma de probabilidades sobre el dataset de entrega
axes[1].hist(proba_pred, bins=40, color="#9b59b6", alpha=0.75, edgecolor="white")
axes[1].axvline(THRESHOLD, color="#2ecc71", linestyle="-", linewidth=2.5,
                label=f"Umbral óptimo ({THRESHOLD:.2f})")
axes[1].axvline(0.50, color="#95a5a6", linestyle="--", linewidth=1.8,
                label="Umbral default (0.50)")
axes[1].set_xlabel("Probabilidad predicha (clase 1 = fumador)", fontsize=10)
axes[1].set_ylabel("Cantidad de registros", fontsize=10)
axes[1].set_title("Distribución de probabilidades — dataset de entrega",
                  fontsize=11, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle(
    "Verificación de predicciones sobre el dataset de entrega",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 5. Generación del archivo de entrega

In [ ]:
df_output = pd.DataFrame({
    "ID":                 IDs.values,
    "smoking_prediction": y_pred
})

# Verificaciones finales
assert df_output["ID"].nunique() == len(df_output),      "IDs duplicados"
assert set(df_output["smoking_prediction"].unique()).issubset({0, 1}), "Valores distintos de 0/1"
assert len(df_output) == len(df_predict),                "Filas faltantes"

print("Verificaciones:")
print(f"  ✓ IDs únicos: {df_output['ID'].nunique()}")
print(f"  ✓ Valores en smoking_prediction: {sorted(df_output['smoking_prediction'].unique())}")
print(f"  ✓ Total de filas: {len(df_output)}")
print()
print("Primeras 10 filas:")
print(df_output.head(10).to_string(index=False))

In [ ]:
output_path = PATH_PROC / "smoking_predictions.csv"
df_output.to_csv(output_path, index=False)

print(f"✓ Archivo generado: {output_path}")
print(f"  Tamaño: {output_path.stat().st_size / 1024:.1f} KB")
print(f"  Filas: {len(df_output)}")
print(f"  Columnas: {list(df_output.columns)}")

## 6. Resumen de la predicción

### Modelo utilizado

| Campo | Valor |
|---|---|
| Archivo | `models/xgboost_optimized.joblib` |
| Algoritmo | XGBoost (XGBClassifier) |
| `n_estimators` | 100 |
| `max_depth` | 6 |
| `learning_rate` | 0.1 |
| `subsample` | 0.8 |
| `colsample_bytree` | 0.8 |
| F1-Score (test, umbral 0.50) | 0.6947 |
| F1-Score (test, umbral óptimo ~0.36) | 0.7223 |

### Por qué este modelo

Se entrenaron dos modelos durante el proceso: un **baseline** y un **optimizado** (GridSearchCV).
El GridSearchCV confirmó que los parámetros del baseline son los óptimos dentro del espacio explorado,
por lo que ambos modelos tienen los mismos hiperparámetros.

Se eligió **`xgboost_optimized`** porque es el resultado del proceso de validación cruzada (cv=3)
y representa la decisión final del pipeline de optimización.

### Por qué el umbral ajustado

El análisis de `05_validacion.ipynb` mostró que el umbral default (0.50) no es el óptimo para F1.
Bajando el umbral a ~0.36 se detectan más fumadores reales (recall sube de 0.72 a ~0.89)
con una mejora neta de +2.7 pp en F1, que es la métrica objetivo del TP.

### Trade-off del umbral sobre la distribución predicha

| Umbral | F1-Test | Fumadores predichos (entrega) | Fumadores reales (train) |
|---|---|---|---|
| 0.50 (default) | 0.6947 | 39.0% | 36.7% |
| **~0.36 (óptimo F1)** | **0.7223** | **53.5%** | **36.7%** |

Al bajar el umbral, la proporción predicha supera a la real porque el modelo acepta más falsos positivos
para capturar más fumadores verdaderos (alto recall). Para la métrica objetivo del TP (F1-Score),
el umbral ~0.36 es la mejor elección.

### Resultado

| Campo | Valor |
|---|---|
| Total de registros a predecir | 5,692 |
| Predichos como fumadores (1) | 3,048 (53.5%) |
| Predichos como no fumadores (0) | 2,644 (46.5%) |
| Archivo de salida | `data/processed/smoking_predictions.csv` |
| Columnas del archivo | `ID`, `smoking_prediction` (valores: 0 ó 1) |